# 🎯 Query Enhancement: 
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

### Types of Query Enhancements:
- [1. Query Expansion Technique](#1-query-expansion-technique)
- [2. Query Decomposition](#2-query-decomposition)
- [3. HyDE](#3-hyde)

### 1. Query Expansion Technique:
In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be. That's where Query Expansion becomes very handy.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [8]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [ ]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [4]:
for i, chunk in enumerate(chunks[:2]):
    print(f"\n📄Chunk {i+1}:")
    print(f"    Source: {chunk.metadata['source']}")
    print(f"    Length: {len(chunk.page_content)} characters")
    print(f"    Content: {chunk.page_content[:100]}...")  # Print first 100 characters
    print("-" * 40)


📄Chunk 1:
    Source: langchain_crewai_dataset.txt
    Length: 299 characters
    Content: LangChain is an open-source framework designed for developing applications powered by large language...
----------------------------------------

📄Chunk 2:
    Source: langchain_crewai_dataset.txt
    Length: 172 characters
    Content: and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LL...
----------------------------------------


In [13]:
### step 2: Vector Store
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [14]:
### step 3: MMR Retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5})

In [15]:
## step 4 : LLM and Prompt
llm=init_chat_model("openai:o4-mini")

In [16]:
# Query expansion prompt
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")
query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()

In [17]:
# test the query expansion chain
query_expansion_chain.invoke({"query":"Langchain memory"})

'Expanded query:  \nLangChain memory OR “Lang Chain” memory OR “LangChain state management” OR “LangChain context persistence” OR “conversation memory” OR “dialogue memory” OR “session memory” OR “context buffer” OR “memory buffer” OR “memory retriever” OR “memory store” OR “memory module” OR “embeddings-based memory” OR “vectorstore memory” OR “persistent memory” OR “ephemeral memory” OR “summary memory” OR “long-term memory” OR “short-term memory” OR “ConversationBufferMemory” OR “ConversationSummaryMemory” OR “CombinedMemory” OR “chatbot memory” OR “pipeline memory” OR “LLM memory interface” OR “external memory adapter” OR “contextual recall” OR “stateful LLM calls” OR “memory management in LangChain”'

In [18]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = answer_prompt | llm

In [19]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"], # pyright: ignore[reportIndexIssue]
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]})) # pyright: ignore[reportIndexIssue]
    })
    | document_chain
)

In [20]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:

(LangChain OR “Lang Chain”) AND (memory OR “memory types” OR “memory support” OR “context retention” OR “state management” OR “conversation memory” OR “chat memory” OR “session persistence” OR “buffer memory” OR “chat history” OR “vector store memory” OR “retrieval memory” OR “summary memory” OR “external memory store” OR “custom memory module” OR “in‐memory storage” OR “long‐term memory” OR “short‐term memory” OR “episodic memory”)  
AND (“memory architecture” OR “stateful agents” OR “context window” OR “conversation context” OR “LLM memory” OR “RAG” OR “retrieval‐augmented generation” OR Redis OR FAISS OR Pinecone OR “SQLStore” OR “MongoDB” OR “memory backend”)  

Optional terms for broader coverage:  
• synonyms: “context persistence”, “conversation state”  
• technical descriptors: “memory buffer”, “chain-of-thought memory”, “summary-based memory”  

This expanded query will surface documentation, tutorials, and code examples describing all the different memory mod

In [21]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:

("CrewAI" OR "Crew AI" OR "Crew Intelligence AI") AND  
("agents" OR "AI agents" OR "autonomous agents" OR "software agents" OR "intelligent assistants") AND  
("crew scheduling" OR "crew management" OR "workforce planning" OR "roster optimization" OR "dispatching") AND  
("machine learning" OR "reinforcement learning" OR "multi-agent system" OR "constraint programming" OR "predictive analytics" OR "optimization algorithms" OR "NLP") AND  
(aviation OR maritime OR shipping OR rail OR hospitality OR event staffing OR manufacturing)  

Synonyms & related terms to sprinkle throughout:  
• “autonomous crew scheduling bots”  
• “intelligent crew planning assistants”  
• “AI-driven workforce rostering”  
• “crew operations optimization”  
• “automated shift-assignment system”  
• “multi-agent reinforcement learning for crew rostering”  

Use these variants in document searches to surface papers, product docs or technical blogs on AI-powered crew management and scheduling sys

### 2. Query Decomposition:
🧠 Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [24]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [ ]:
### step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [ ]:
### step 2: Vector Store and retriever
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

In [27]:
### step 3: LLM and Prompt
llm=init_chat_model(model="groq:gemma2-9b-it")

In [28]:
# Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [29]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})

In [30]:
print(decomposition_question)

Here are some sub-questions to decompose the complex question:

1. **How does LangChain implement memory in its models?**
2. **What are the different types of agents supported by LangChain, and how do they utilize memory?**
3. **How does CrewAI handle memory management in its models?** 
4. **What are the key differences in agent capabilities and memory integration between LangChain and CrewAI?** 


These sub-questions break down the comparison into specific aspects of memory and agent functionality in each framework, allowing for more targeted document retrieval. 



In [31]:
# QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [32]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):

    # Step 1: Decompose the query into sub-questions
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]

    # Step 2: Retrieve documents and get answers for each sub-question
    results = []
    for sub_q in sub_questions:
        docs = retriever.invoke(sub_q)
        result = qa_chain.invoke({"input": sub_q, "context": docs})
        results.append(f"Q: {sub_q}\nA: {result}")

    # Step 3: Combine answers into a final response
    final_answer = "\n\n".join(results)
    return final_answer


In [33]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: Here are some sub-questions that break down the complex query:
A: Please provide the complex query you want to break down into sub-questions. 

I need the main question to help you formulate the sub-questions effectively. 

For example, you could say:

"Here is the complex query: **How can CrewAI be used to analyze legal documents?**  Break this down into sub-questions." 


Then I can help you with sub-questions like:

* What are the capabilities of CrewAI that are relevant to legal document analysis?
* What types of legal documents can CrewAI analyze?
* What are some examples of how CrewAI can be used to extract information from legal documents?
* Are there any limitations to using CrewAI for legal document analysis? 



Let me know your main query! 


Q: **What memory mechanisms does LangChain utilize?**
A: LangChain utilizes memory modules like **ConversationBufferMemory** and **ConversationSummaryMemory**. 

These modules allow the LLM to:

* **Maintain awarenes

### 3. HyDE
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short.
2. Language mismatch between query and documents. 
3. You want to retrieve based on answer content, not question words


In [34]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
from langchain_community.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

In [44]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [37]:
# 1. Load and chunk your dataset
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

In [39]:
# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
chunks = text_splitter.split_documents(documents=documents)

In [40]:
for i, chunk in enumerate(chunks[:2]):
    print(f"\n📄Chunk {i+1}:")
    print(f"    Source: {chunk.metadata['source']}")
    print(f"    Length: {len(chunk.page_content)} characters")
    print(f"    Content: {chunk.page_content[:100]}...")  # Print first 100 characters
    print("-" * 40)


📄Chunk 1:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 291 characters
    Content: Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and in...
----------------------------------------

📄Chunk 2:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 213 characters
    Content: of NeXT and chairman and majority shareholder of Pixar. He was a pioneer of the personal computer re...
----------------------------------------


In [43]:
# 2. Build vector store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(documents = chunks,embedding=embedding_model,persist_directory = "output/steve_jobs_for_hyde.db")

##create the retriever
base_retriever=db.as_retriever(search_kwargs = {"k":5})

In [46]:
# 3. Set up the LLM you’ll use to generate hypothetical answers
llm = init_chat_model("groq:gemma2-9b-it")

In [51]:
## Generating a prompt gor generating HyDE
def get_hyde_doc(query) -> str:
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template = template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return str(hypo_doc)

In [50]:
query = 'When was Steve Jobs fired from Apple?'
print(get_hyde_doc(query=query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
## The Steve Jobs Exodus: A Deeper Look at His 1985 Departure from Apple

Steve Jobs' firing from Apple in 1985 is a pivotal moment in both his personal story and the history of the tech giant. It's a narrative often simplified, but the reality is more nuanced.  

**The Catalyst: A Clash of Personalities and Visions**

The seeds of Jobs' departure were sown years earlier. His visionary leadership, while crucial to Apple's early success, also bred friction within the company. His demanding nature, coupled with a fierce desire for control, clashed with the more collaborative and pragmatic approach favoured by Apple's CEO, John Sculley. 

The Macintosh, Jobs' brainchild, faced internal resistance and was met with lukewarm sales, further exacerbating the tension. 

In [52]:
matched_docs = base_retriever.invoke(get_hyde_doc(query))
for i, doc in enumerate(matched_docs):
	print(f"\n📄Matched Document {i+1}:")
	print(f"    Source: {doc.metadata.get('source', doc.metadata.get('title', 'N/A'))}")
	print(f"    Length: {len(doc.page_content)} characters")
	print(f"    Content: {doc.page_content[:200]}...")  # Print first 200 characters
	print("-" * 40)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]

📄Matched Document 1:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs
    Length: 297 characters
    Content: In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer plat...
----------------------------------------

📄Matched Document 2:
    Source: https://en.wikipedia.org/wiki/Steve_Jobs_(film)
    Length: 295 characters
    Content: Apple CEO John Sculley demands to know why the world believes he fired Jobs – Jobs was actually forced out by the Apple board, who were resolute on updating the Apple II following the Macintosh's lack...
----------------------------------------

📄Matched Document 3:
    Source: https://e

#### 3.1. Langchain-HypotheticalDocumentEmbedder

In [58]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [56]:
from langchain.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain

In [59]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [54]:
# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [61]:
# Step 2: Set up LLM and embeddings
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = init_chat_model("groq:gemma2-9b-it")

According to the official documentation and LangChain source code (mapping in PROMPT_MAP), the default options are:

- web_search
- sci_fact
- arguana
- trec_covid
- fiqa
- dbpedia_entity
- trec_news
- mr_tydi

In [62]:
# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key="web_search"
)

In [63]:
# Step 4: Store documents in Chroma with HyDE embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain_hyde"
)

In [64]:
# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [65]:
# Step 6: Final RAG Pipeline
def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    print(matched_docs)
    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })
    return response

In [66]:
# Step 7: Run example query
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("✅ Final Answer:\n", answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v6)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v2)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_cont

#### 3.2. Custom Prompt for Langchain HyDE

In [67]:
from langchain.prompts import PromptTemplate
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)

# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)